# multilingual-e5-large + UMAP + K-Means (auto-k)

Pairs the finalized embedding model, `multilingual-e5-large`, with **K-Means**. Motivated by a
short-text clustering benchmark (arXiv:2511.19350) finding K-Means, given a good cluster-count
estimate, significantly outperforms parameter-light methods like HDBSCAN/OPTICS.

K-Means needs k up front — here we sweep a wide range (10 to ~900, since events in this corpus tend
to form many small clusters) and keep whichever k maximizes silhouette score.

In [1]:
!pip install -q umap-learn

In [2]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_kmeans")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/e5_kmeans


In [3]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [4]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [5]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [6]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [7]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

df shape after text cleaning: (1999, 7)


,article_id,publisher,url,published_at,title,body_text,text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...,"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [9]:
# Build documents for embedding (title + body), with the "passage: " prefix multilingual-e5
# models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text"].iloc[0][:500]

Documents: 1999


'passage: දැන් තෝරු-මෝරු අහුවෙන කාලේ. දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහනුවර දිස්ත්\u200dරික් මන්ත්\u200dරී ජගත් මනුවර්ණ මහතා:-(ජා.ජ.බ) පාර්ලිමේන්තුවේදී පැවසීය.\nදූෂණයට විරුද්ධ වීම භයානක බවත් දූෂණයට විරුද්ධ නොවී සිටීම ඊට වඩා භයානක බවත් අනුර දිසානායක ජනාධිපති\xa0 එක්සත් ජාතීන්ගේ මහා මණ්ඩලයේ අමතමින් ප්\u200dරකාශ කළා.එය අප නැවත අවධාරණය කළ යුතුයි.අපි දේශපාලන පලි ගැනීම් කරනවා යැයි චෝදනා කරනවා.නමුත් ඇත්ත ඒක නෙමෙයි.මත්තල ගුවන් තොටුපොලේ මගින් පර්යන්තය එක\xa0 ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම ඊට පස'

In [10]:
# Load multilingual-e5-large (the finalized embedding model for this project)
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 1024, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-23): 24 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwi

In [11]:
# Mean pooling — established as the best strategy for this model on this corpus in
# notebooks/clustering_e5.ipynb (separation_score 0.2130 for mean vs 0.1264 for max pooling)
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


embeddings = embed_passages(df["passage_text"].tolist(), batch_size=16)
print(embeddings.shape)

(1999, 1024)


In [12]:
# Embedding separation score (comparable across notebooks): 1 - mean pairwise cosine similarity
# on a random 100-document sample of the raw embeddings.
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sample_emb = embeddings[sample_idx]

sims = cosine_similarity(sample_emb)
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()

print(f"Embedding model: {embedding_model_name}")
print(f"mean_sim={pairwise.mean():.4f} std={pairwise.std():.4f} min={pairwise.min():.4f} max={pairwise.max():.4f}")
print(f"Separation score (1 - mean cosine similarity): {separation_score:.4f}")

Embedding model: intfloat/multilingual-e5-large
mean_sim=0.7909 std=0.0243 min=0.7123 max=0.9162
Separation score (1 - mean cosine similarity): 0.2091


In [13]:
# Reduce dimensionality with UMAP (same settings used across this project's UMAP-based
# notebooks, e.g. notebooks/clustering_BGE_M3_BERTopic.ipynb)
from umap import UMAP

umap_model = UMAP(
    n_neighbors=3,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)
reduced_embeddings = umap_model.fit_transform(embeddings)
print(reduced_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(1999, 5)


In [14]:
# Sweep k for K-Means, pick the value that maximizes silhouette score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

candidate_k = sorted(set(list(range(10, 100, 10)) + list(range(100, 400, 25)) + list(range(400, 900, 50))))
sweep_results = []

for k in candidate_k:
    if k >= len(reduced_embeddings):
        continue
    model = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = model.fit_predict(reduced_embeddings)
    sil = silhouette_score(reduced_embeddings, labels, metric="cosine")
    sweep_results.append((k, sil))
    print(f"k={k:>4}  silhouette={sil:.4f}")

sweep_df = pd.DataFrame(sweep_results, columns=["k", "silhouette"]).sort_values("silhouette", ascending=False)
sweep_df.head(10)

k=  10  silhouette=0.4869
k=  20  silhouette=0.5843
k=  30  silhouette=0.5995
k=  40  silhouette=0.6089
k=  50  silhouette=0.6176
k=  60  silhouette=0.6360
k=  70  silhouette=0.6912
k=  80  silhouette=0.7353
k=  90  silhouette=0.7361
k= 100  silhouette=0.7563
k= 125  silhouette=0.7634
k= 150  silhouette=0.7786
k= 175  silhouette=0.7661
k= 200  silhouette=0.7613
k= 225  silhouette=0.7550
k= 250  silhouette=0.7841
k= 275  silhouette=0.7776
k= 300  silhouette=0.7847
k= 325  silhouette=0.7854
k= 350  silhouette=0.7921
k= 375  silhouette=0.8015
k= 400  silhouette=0.7977
k= 450  silhouette=0.7928
k= 500  silhouette=0.7833
k= 550  silhouette=0.7712
k= 600  silhouette=0.7659
k= 650  silhouette=0.7494
k= 700  silhouette=0.7306
k= 750  silhouette=0.7086
k= 800  silhouette=0.6876
k= 850  silhouette=0.6708


,k,silhouette
20,375,0.801486
21,400,0.797744
22,450,0.792837
19,350,0.792059
18,325,0.785444
17,300,0.784698
15,250,0.784076
23,500,0.783281
11,150,0.778595
16,275,0.777595


In [15]:
# Refit at the best k found
best_k = int(sweep_df.iloc[0]["k"])
print(f"Best k: {best_k}")

model = KMeans(n_clusters=best_k, random_state=SEED, n_init=10)
df["cluster_id"] = model.fit_predict(reduced_embeddings)

print(df["cluster_id"].value_counts().head(20))
print("Number of clusters found:", df["cluster_id"].nunique())

extra_row_fields = {"best_k": best_k}

Best k: 375
cluster_id
70     17
93     16
144    15
95     15
66     15
100    13
56     13
73     12
11     12
43     11
18     11
183    11
205    11
337    11
4      10
228    10
26     10
257    10
13     10
160    10
Name: count, dtype: int64
Number of clusters found: 375


In [16]:
# Clustering evaluation
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

labels = df["cluster_id"].values
mask = labels != -1  # -1 = noise; only meaningful for density-based algorithms
X_valid = reduced_embeddings[mask]
labels_valid = labels[mask]
n_clusters = len(set(labels_valid))
noise_ratio = 1 - mask.mean()

print(f"Model: {embedding_model_name} + UMAP + KMeans")
print(f"Articles: {len(df)} | Clusters (excl. noise): {n_clusters} | Noise ratio: {noise_ratio:.2%}")

if n_clusters > 1:
    sil = silhouette_score(X_valid, labels_valid, metric="cosine")
    dbi = davies_bouldin_score(X_valid, labels_valid)
    ch = calinski_harabasz_score(X_valid, labels_valid)
    print(f"Silhouette Score (cosine): {sil:.4f}")
    print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
    print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")
else:
    sil = dbi = ch = float("nan")
    print("Not enough clusters to compute silhouette/DBI/CH.")

scores_path = RESULTS_DIR / "e5_kmeans_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "UMAP+KMeans",
    "embedding_dim": embeddings.shape[1],
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": round(noise_ratio, 4),
    "silhouette": round(sil, 4) if n_clusters > 1 else None,
    "davies_bouldin": round(dbi, 4) if n_clusters > 1 else None,
    "calinski_harabasz": round(ch, 2) if n_clusters > 1 else None,
}
row.update(extra_row_fields)
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

Model: intfloat/multilingual-e5-large + UMAP + KMeans
Articles: 1999 | Clusters (excl. noise): 375 | Noise ratio: 0.00%
Silhouette Score (cosine): 0.8015
Davies-Bouldin Index:      0.4159  (lower is better)
Calinski-Harabasz Index:   13603.78  (higher is better)
Saved scores to /kaggle/working/results/e5_kmeans/e5_kmeans_scores.csv


In [17]:
# Inspect sample titles per cluster
for cluster_id, group in list(df[df["cluster_id"] != -1].groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

=== Cluster 0 (5 articles) ===
- වසර 45ක් ස්වේච්ඡාවෙන් දරුවන්ට පාර කියන අත්තම්මා
- පතල් වැඩ නැති දාට කැලේ යනවා දර කපන්න. මට ඕනා ළමයව හොඳ තැනකට ගෙනියන්න.'
- මිනිස්සු වෙනුවෙන්, භූමිය වෙනුවෙන් අපි සටන් වැදුණේ. අපිට මොනවාද ලැබුණේ ?'
- නොමිලේ දැනුම බෙදන මහියංගණයේ උපකාරක පන්ති ගුරුතුමා : 'මම කරන්නේ පිං අතේ වැඩක් නොවෙයි'
- අධ්‍යාපනයේ මහිමයෙන් ලෝකය ජය ගත් හේනේ පැලක පාඩම් කළ ශ්‍රී ලාංකිකයා

=== Cluster 1 (5 articles) ===
- විදෙස්ගත ශ්‍රමිකයින්ට මෙරටට භාණ්ඩ රැගෙන ඒමේදී බලාත්මක කර ඇති කොන්දේසි මෙන්න
- සහල් අර්බුදය: ආනයනික සහල් තොග දිවයිනට
- සහල් මෙට්‍රික්ටොන් 67,000ක් මෙරටට ආනයනයකර අවසන්
- වී නිෂ්පාදනය වැඩිවෙලා සහල් ආනයන වියදම අඩුවෙයි
- සහල් ආනයනයට අවසර දුන් කාලය තවත් පැය කිහිපයකින් අවසන්: ජනපති ඇතුළු රජය මෙතෙක් ගත් පියවර මොනවා ද?

=== Cluster 2 (2 articles) ===
- ඩොලරය ගැන රනිල් අනතුරු අඟවයි
- මේක L බෝඩ් ආණ්ඩුවක් - රනිල්

=== Cluster 3 (6 articles) ===
- බුල්නෑව නිවසක් බිඳ ජීප් රථයක්, රන් භාණ්ඩ හා මුදල් කෝල්ල කෑ සැකකරුවන් අල්ලයි
- ග්‍රෑන්ඩ්පාස් වෙඩි තැබීමට සැක දෙදෙනෙකු අල්ලයි
- රත්‍රං බවට සැක පු

In [18]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

=== Cluster 336 (2 articles) ===
- ‘Visa ශ්‍රී ලංකා සයිබර් ආරක්ෂණ සමුළුව 2026’ සාර්ථකව නිමාවේ
- 2026 අයවැයෙන් ඩිජිටල්කරණය වන ක්ෂේත්‍ර මොනවා ද? මුදල් වෙන් වෙන්නේ කෙසේ ද?

=== Cluster 318 (4 articles) ===
- මැණික්වලට අඟරු චෙක් දුන් පහක් අත්අඩංගුවට
- රජමහ වෙහෙරක  තිබූ මැණික්  සොරු අරන්
- නිධන් සොයන්න ගිය තිදෙනකු පොලිස් දැලේ
- පොලොව බිඳින ස්කෑනරයක් සමග දෙකක් මාට්ටු

=== Cluster 67 (2 articles) ===
- ව්‍යාජ ප්‍රතිදේහ නිෂ්පාදනය කළ කර්මාන්ත ශාලාවෙන් හමුවූ දේ
- මිද්දෙණියේ ඉඩමෙන් පොලිස් නිලධාරීන් රාජකාරි සඳහා භාවිතා කරන උපාංග හමුවෙයි

=== Cluster 218 (4 articles) ===
- දිසා විනිසුරු නිල නිවසේ රාජකාරීය අතරතුරදී  සූර් පිට සිටි පී සී අත්අඩංගුවට
- වැඩ තහනම් වූ පීසී කෝටි 8ක මත්ද්‍රව්‍ය සමඟ මාට්ටු
- හෙරොයින් සමඟ  හිටපු පීසී අල්ලයි
- විනිසුරුවරයෙකුගේ නිවසකටත් හොරු පනී

=== Cluster 175 (5 articles) ===
- මන්දකට හසු වී සිටි කදුකර කොටියා නිර්වින්දනය කර මුදා ගනිියි
- පැරණි පොලිස් මූලස්ථානයේ තිබූ CCTV කැමරා 7ක් අතුරුදන්
- අධිවේගයේ කඩා වැටුණ කොන්ක්‍රීට් ආධාරකය පොලීසියේ පරීක්ෂණයට
- මවගෙන් වෙන් කෙරුණු වන අලි 

In [19]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_kmeans_assignments.csv"
df.drop(columns=["passage_text"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")

Saved assignments: /kaggle/working/results/e5_kmeans/e5_kmeans_assignments.csv


In [20]:
# Print every cluster (capped at 20 titles each, so this stays readable even when there are
# hundreds of clusters)
for cluster_id, group in sorted(
    df[df["cluster_id"] != -1].groupby("cluster_id"), key=lambda item: item[0]
):
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(20):
        print(f"- {title}")
    if len(group) > 20:
        print(f"... ({len(group) - 20} more)")
    print()


=== Cluster 0 (5 articles) ===
- වසර 45ක් ස්වේච්ඡාවෙන් දරුවන්ට පාර කියන අත්තම්මා
- පතල් වැඩ නැති දාට කැලේ යනවා දර කපන්න. මට ඕනා ළමයව හොඳ තැනකට ගෙනියන්න.'
- මිනිස්සු වෙනුවෙන්, භූමිය වෙනුවෙන් අපි සටන් වැදුණේ. අපිට මොනවාද ලැබුණේ ?'
- නොමිලේ දැනුම බෙදන මහියංගණයේ උපකාරක පන්ති ගුරුතුමා : 'මම කරන්නේ පිං අතේ වැඩක් නොවෙයි'
- අධ්‍යාපනයේ මහිමයෙන් ලෝකය ජය ගත් හේනේ පැලක පාඩම් කළ ශ්‍රී ලාංකිකයා

=== Cluster 1 (5 articles) ===
- විදෙස්ගත ශ්‍රමිකයින්ට මෙරටට භාණ්ඩ රැගෙන ඒමේදී බලාත්මක කර ඇති කොන්දේසි මෙන්න
- සහල් අර්බුදය: ආනයනික සහල් තොග දිවයිනට
- සහල් මෙට්‍රික්ටොන් 67,000ක් මෙරටට ආනයනයකර අවසන්
- වී නිෂ්පාදනය වැඩිවෙලා සහල් ආනයන වියදම අඩුවෙයි
- සහල් ආනයනයට අවසර දුන් කාලය තවත් පැය කිහිපයකින් අවසන්: ජනපති ඇතුළු රජය මෙතෙක් ගත් පියවර මොනවා ද?

=== Cluster 2 (2 articles) ===
- ඩොලරය ගැන රනිල් අනතුරු අඟවයි
- මේක L බෝඩ් ආණ්ඩුවක් - රනිල්

=== Cluster 3 (6 articles) ===
- බුල්නෑව නිවසක් බිඳ ජීප් රථයක්, රන් භාණ්ඩ හා මුදල් කෝල්ල කෑ සැකකරුවන් අල්ලයි
- ග්‍රෑන්ඩ්පාස් වෙඩි තැබීමට සැක දෙදෙනෙකු අල්ලයි
- රත්‍රං බවට සැක පු